# 00_configure_workshop (Voila)

Set machine-specific paths before running the workshop:
- **rockem-suite** checkout (binaries and Python package)
- MPI launcher (`mpirun`)
- Default MPI process count
- Default SEG-Y model

Settings are saved to `workshop_config.json` at the repo root.


In [ ]:
from pathlib import Path
import json
import os
import signal
import sys

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.modules.workshop_config import (
    WorkshopConfig,
    _defaults,
    config_path,
    load_config,
    save_config,
    validate_config,
)

try:
    import ipywidgets as ipw
    from IPython.display import display
except Exception as exc:
    raise RuntimeError(
        'Missing GUI dependencies. Install with: pip install voila ipywidgets plotly numpy ipykernel'
    ) from exc

VOILA_PID_FILE = ROOT / '.voila_configure_server.pid'

cfg = load_config(ROOT)

rockem_input = ipw.Text(
    value=str(cfg.rockem_suite_root),
    description='rockem-suite:',
    layout=ipw.Layout(width='95%'),
    style={'description_width': '120px'},
)
mpirun_input = ipw.Text(
    value=cfg.mpirun,
    description='mpirun:',
    layout=ipw.Layout(width='400px'),
    style={'description_width': '120px'},
)
nproc_input = ipw.BoundedIntText(
    value=int(cfg.nproc_default),
    min=1,
    max=max(2, os.cpu_count() or 2),
    description='nproc:',
    layout=ipw.Layout(width='220px'),
    style={'description_width': '120px'},
)
segy_input = ipw.Text(
    value=str(cfg.default_segy),
    description='default SEG-Y:',
    layout=ipw.Layout(width='95%'),
    style={'description_width': '120px'},
)
workspace_input = ipw.Text(
    value=cfg.workspace_dir,
    description='workspace:',
    layout=ipw.Layout(width='300px'),
    style={'description_width': '120px'},
)

status_out = ipw.Textarea(value='', description='Status', layout=ipw.Layout(width='95%', height='180px'))
validation_out = ipw.HTML(value='<i>Click Validate to check settings.</i>')


def _current_config() -> WorkshopConfig:
    segy = Path(segy_input.value.strip()).expanduser()
    if not segy.is_absolute():
        segy = (ROOT / segy).resolve()
    return WorkshopConfig(
        rockem_suite_root=Path(rockem_input.value.strip()).expanduser().resolve(),
        mpirun=mpirun_input.value.strip() or 'mpirun',
        nproc_default=int(nproc_input.value),
        default_segy=segy,
        workspace_dir=workspace_input.value.strip() or 'workspace',
        _root=ROOT,
    )


def on_validate(_):
    current = _current_config()
    os.environ['ROCKEM_SUITE_ROOT'] = str(current.rockem_suite_root)
    rows = []
    for name, ok, detail in validate_config(current):
        color = '#006400' if ok else '#aa0000'
        mark = 'OK' if ok else 'FAIL'
        rows.append(f'<div><b style="color:{color}">{mark}</b> {name}: <code>{detail}</code></div>')
    validation_out.value = ''.join(rows)
    status_out.value = 'Validation finished.'


def on_save(_):
    current = _current_config()
    path = save_config(current, ROOT)
    status_out.value = f'Saved configuration to {path}'
    on_validate(None)


def on_reset(_):
    defaults = _defaults(ROOT)
    rockem_input.value = str(defaults.rockem_suite_root)
    mpirun_input.value = defaults.mpirun
    nproc_input.value = defaults.nproc_default
    segy_input.value = str(defaults.default_segy)
    workspace_input.value = defaults.workspace_dir
    status_out.value = 'Reset to defaults (not saved yet).'


def on_quit(_):
    if VOILA_PID_FILE.exists():
        try:
            pid = int(VOILA_PID_FILE.read_text().strip())
            os.kill(pid, signal.SIGTERM)
            status_out.value = f'Sent SIGTERM to Voila server (pid={pid}).'
        except Exception as exc:
            status_out.value = f'Could not stop Voila server: {exc}'
    else:
        status_out.value = 'No Voila PID file found.'


validate_btn = ipw.Button(description='Validate', button_style='primary')
save_btn = ipw.Button(description='Save', button_style='success')
reset_btn = ipw.Button(description='Reset defaults')
quit_btn = ipw.Button(description='Quit Voila')

validate_btn.on_click(on_validate)
save_btn.on_click(on_save)
reset_btn.on_click(on_reset)
quit_btn.on_click(on_quit)

display(
    ipw.VBox([
        ipw.HTML('<p>Configure paths for this machine, then click <b>Save</b>. Other notebooks read <code>workshop_config.json</code> at startup.</p>'),
        rockem_input,
        mpirun_input,
        nproc_input,
        segy_input,
        workspace_input,
        ipw.HBox([validate_btn, save_btn, reset_btn, quit_btn]),
        validation_out,
        status_out,
    ])
)

if config_path(ROOT).exists():
    status_out.value = f'Loaded existing config from {config_path(ROOT)}'
else:
    status_out.value = 'No workshop_config.json yet — using defaults.'
